<a href="https://colab.research.google.com/github/Satyaharshith/Agentic-AI-Learning/blob/main/Day_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
!pip install -q langchain langchain-groq

In [40]:

import os
from google.colab import userdata

# Get API key securely from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Import LangChain Groq
from langchain_groq import ChatGroq

# Initialize AI model
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Groq connected successfully!")


Groq connected successfully!


In [41]:

response = llm.invoke("What is Artificial Intelligence?")

print(response.content)


**Artificial Intelligence (AI)** is the field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include:

| Category | Typical Tasks |
|----------|---------------|
| **Perception** | Vision, speech, and sensor data interpretation |
| **Reasoning & Decision‑Making** | Planning, problem solving, game playing |
| **Learning** | Pattern recognition, adaptation from data |
| **Interaction** | Natural language understanding, dialogue, robotics |

---

### Core Concepts

| Concept | What It Means |
|---------|---------------|
| **Intelligence** | The ability to acquire knowledge, adapt to new situations, and solve problems. |
| **Artificial** | Implemented by machines (software, hardware, or both). |
| **Goal‑Oriented** | Systems are designed to achieve specific objectives (e.g., classify images, recommend products). |

---

### Historical Milestones

| Year | Milestone |
|------|-----------|
| 1950 | Turing’

In [42]:
SYSTEM = """
ROLE:
You are a friendly customer support assistant
for Sweet Crumbs Home Bakery.

FACTS:
- Plum cake costs Rs 500.
- Chocolate cake costs Rs 650.
- Cupcake costs Rs 80 each.
- All cakes have eggless versions.
- Orders require 24 hours notice.

RULES:
- Never invent product prices.
- Never promise delivery times or locations.
- For information not listed in FACTS, reply:
  "I'll check with the owner."
- Calculate totals correctly.
- For multiple cupcakes, multiply Rs 80 by quantity.

FORMAT:
- Keep answers under 50 words.
- Use simple, friendly English.
"""

In [43]:

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

print("Prompt template created successfully!")


Prompt template created successfully!


In [44]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

chain = prompt | llm | StrOutputParser()

print("Temperature changed to 1.0")

Temperature changed to 1.0


In [45]:

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print(ask("Hi, I am Ravi. Do you make eggless cakes?"))

print(ask("What is my name?"))


Hi Ravi! Yes, all our cakes—including plum, chocolate, and cupcakes—are available in eggless versions. Let me know which one you’d like to order!
I'll check with the owner.


In [46]:
%%writefile tests.csv
question,must_include
What cakes are available?,plum cake
What is the price of chocolate cake?,650
Do you have eggless cakes?,eggless
Can I order a cake for tomorrow?,24 hours
Do you provide home delivery?,check with the owner
What is the price of six cupcakes?,480
Can I customize a birthday cake?,check with the owner
What flavours are available?,check with the owner
How can I place an order?,check with the owner
Can I cancel or modify my order?,check with the owner

Overwriting tests.csv


In [47]:
import csv

def run_tests(path="tests.csv"):
    with open(path, newline="", encoding="utf-8") as file:
        tests = list(csv.DictReader(file))

    passed = 0

    for test in tests:
        answer = chain.invoke({
            "history": [],
            "question": test["question"]
        })

        expected = test["must_include"]
        success = expected.lower() in answer.lower()

        if success:
            passed += 1
            status = "PASS"
        else:
            status = "FAIL"

        print(f"{status} | {test['question']}")
        print(f"Answer: {answer}\n")

    total = len(tests)
    print(f"Result: {passed}/{total} passed ({passed / total:.0%})")

run_tests()

PASS | What cakes are available?
Answer: We have plum cake, chocolate cake, and cupcakes. All are egg‑less. Each cupcake is Rs 80. Enjoy!

PASS | What is the price of chocolate cake?
Answer: The chocolate cake costs Rs 650.

PASS | Do you have eggless cakes?
Answer: Yes, all our cakes—including plum, chocolate, and cupcakes—are available in eggless versions. Let me know what you’d like!

FAIL | Can I order a cake for tomorrow?
Answer: I can’t guarantee tomorrow, as we need 24‑hour notice for orders. Let me know which cake and how many you’d like, and I’ll set it up for you!

PASS | Do you provide home delivery?
Answer: I’ll check with the owner.

PASS | What is the price of six cupcakes?
Answer: Six cupcakes cost 6 × Rs 80 = Rs 480.

PASS | Can I customize a birthday cake?
Answer: Sure! I’ll check with the owner to see what customization options are available and let you know.

PASS | What flavours are available?
Answer: I’ll check with the owner.

FAIL | How can I place an order?
Answ

In [48]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

print("Agent created successfully!")

Agent created successfully!


In [49]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)

Yes, all our cakes have eggless versions.


In [54]:
MENU = {
    "plum cake": 800,
    "chocolate cake": 650,
    "cupcake": 800
}

def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for an item and quantity."""
    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item not found in menu")

    if qty < 1:
        raise ValueError("Quantity must be at least 1")

    return MENU[item] * qty

print(get_price("cupcake", 10))
print(get_price("plum cake"))

8000
800


In [51]:
def is_available(item: str, hours_notice: int) -> bool:
    """Check whether a menu item can be ordered with the given notice."""
    item = item.lower().strip()

    if item not in MENU:
        return False

    return hours_notice >= 24

print(is_available("plum cake", 48))
print(is_available("chocolate cake", 5))

True
False
